# Stage 2: Vote Scaling Analysis - All Models

This notebook extends the vote scaling analysis to all four model architectures.
The goal is to understand how prediction performance scales with the number of GPT-5.2
synthetic votes for each model type, and whether different architectures benefit
differently from more votes.

Design decisions:

- The full 200k GPT vote set is subsampled at seven thresholds: 20k, 40k, 60k, 80k, 100k, 150k and 200k votes.
- For each threshold, a TrueSkill ranking is computed from scratch using the same pipeline as Stage 1.
- All four models are evaluated: Deeper MLP, XGBoost, LightGBM and CatBoost.
- Tree model hyperparameters are tuned once on the full 200k ranking and reused across all thresholds.
- The MLP uses the same fixed architecture as Stage 2 with no additional tuning.
- All other conditions are identical to Stage 2: same feature set, same Pearson threshold,
  same sigma weighting, same 3x5 location-level GroupKFold protocol.
- Votes are sampled by taking the first N rows from the combined 200k dataframe,
  preserving collection order so each threshold is a strict subset of the next.

Vote thresholds: 20k, 40k, 60k, 80k, 100k, 150k, 200k

Models: Deeper MLP, XGBoost, LightGBM, CatBoost

Expected runtime on Kaggle P100: 4 to 6 hours (MLP is the slowest component).

## 1. Install dependencies

In [ ]:
import sys, subprocess
print('Installing dependencies...')
for pkg in ['xgboost', 'lightgbm', 'catboost', 'trueskill']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=True)
print('Done.')

## 2. Imports

In [ ]:
print('Loading libraries...')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import re
import os
import sklearn
import trueskill
warnings.filterwarnings('ignore')
from scipy.stats import pearsonr

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score, make_scorer
from sklearn.utils.class_weight import compute_class_weight

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

SEED = 42
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)
print('Imported.')

## 3. Configuration

All settings are identical to Stage 2.
Upload both 100k GPT vote files and the features file to Kaggle before running.

In [ ]:
# ============================================================
# PATHS
# ============================================================
KAGGLE_INPUT    = '/kaggle/input/datasets/lsestudent/insumos2'
FEATURES_PATH   = f'{KAGGLE_INPUT}/processed_features_merged_02_06_26.csv'
GPT_100K_PATH_1 = f'{KAGGLE_INPUT}/gpt52_votes_100k_20_06_26.csv'
GPT_100K_PATH_2 = f'{KAGGLE_INPUT}/gpt52_votes_100k_2_20_06_26.csv'

# ============================================================
# VOTE SCALING THRESHOLDS
# ============================================================
VOTE_THRESHOLDS = [20_000, 40_000, 60_000, 80_000, 100_000, 150_000, 200_000]

# ============================================================
# SETTINGS — identical to Stage 2
# ============================================================
SIGMA_WEIGHTING        = '1/sigma^2'
PEARSON_THRESHOLD_GPT  = 0.03
NUM_BINS               = 3
BINNING_STRATEGY       = 'qcut'
BIN_LABELS             = ['Low', 'Medium', 'High']
N_SPLITS               = 5
N_PARTITIONS           = 3
VAL_SIZE_WITHIN_TRAIN  = 0.15
EPOCHS                 = 150
BATCH_SIZE             = 64
PATIENCE               = 20
N_ITER_SEARCH          = 12
CV_SEARCH              = 3
IMAGE_PIXELS           = 640 * 640

# colours for chart — one per model
FONT_FAMILY  = 'DejaVu Sans'
plt.rcParams['font.family'] = FONT_FAMILY
MODEL_COLORS = {
    'Deeper MLP': '#e67e22',
    'XGBoost'   : '#8e44ad',
    'LightGBM'  : '#2171B5',
    'CatBoost'  : '#1B7837',
}

print('Configuration loaded.')
print('=' * 60)
print(f'  Features path    : {FEATURES_PATH}')
print(f'  GPT 100k #1      : {GPT_100K_PATH_1}')
print(f'  GPT 100k #2      : {GPT_100K_PATH_2}')
print(f'  Vote thresholds  : {VOTE_THRESHOLDS}')
print(f'  Models           : Deeper MLP, XGBoost, LightGBM, CatBoost')
print(f'  Bins             : {NUM_BINS} ({", ".join(BIN_LABELS)})')
print(f'  Binning strategy : {BINNING_STRATEGY}')
print(f'  Pearson GPT      : |r| > {PEARSON_THRESHOLD_GPT}')
print(f'  Sigma weighting  : {SIGMA_WEIGHTING}')
print(f'  CV               : {N_PARTITIONS} x {N_SPLITS} = {N_PARTITIONS*N_SPLITS} evals per model')
print(f'  Random baseline  : {1/NUM_BINS*100:.2f}%')

## 4. Load datasets

In [ ]:
def normalise_image_name(path: str) -> str:
    return os.path.basename(str(path).strip())


def load_gpt_file(path, label):
    df = pd.read_csv(path)
    if 'model_id' in df.columns and 'user_id' not in df.columns:
        df = df.rename(columns={'model_id': 'user_id'})
    df = df.dropna(subset=['vote', 'image_name1', 'image_name2']).copy()
    df['image_name1'] = df['image_name1'].apply(normalise_image_name)
    df['image_name2'] = df['image_name2'].apply(normalise_image_name)
    df['vote']        = df['vote'].astype(int)
    df['source']      = label
    print(f'  {label:<12}: {len(df):>8,} rows  |  draw rate: {(df["vote"]==0).mean()*100:.2f}%')
    return df[['user_id', 'image_name1', 'image_name2', 'vote', 'source']].copy()


print('Loading GPT vote files...')
df_gpt_100k_1 = load_gpt_file(GPT_100K_PATH_1, 'gpt_100k_1')
df_gpt_100k_2 = load_gpt_file(GPT_100K_PATH_2, 'gpt_100k_2')

df_gpt_200k = pd.concat([df_gpt_100k_1, df_gpt_100k_2], ignore_index=True)
print(f'\nGPT 200k combined:')
print(f'  Total rows : {len(df_gpt_200k):>8,}')
print(f'  Draw rate  : {(df_gpt_200k["vote"]==0).mean()*100:.2f}%')

print('\nLoading features...')
raw_features_df = pd.read_csv(FEATURES_PATH)
print(f'  Features   : {raw_features_df.shape[0]:,} rows x {raw_features_df.shape[1]:,} columns')

## 5. Composite feature construction

Identical to Stage 2. All 14 composites constructed before cleaning.

In [ ]:
def safe_sum(df_, cols):
    present = [c for c in cols if c in df_.columns]
    missing = [c for c in cols if c not in df_.columns]
    if missing:
        print(f'  WARNING: missing columns skipped: {missing}')
    return df_[present].fillna(0).sum(axis=1) if present else pd.Series(0, index=df_.index)


df_with_composites = raw_features_df.copy()

df_with_composites['comp_cycling_infra']      = safe_sum(df_with_composites, ['seg_Bike Lane', 'seg_Bike Rack', 'seg_Bicycle', 'seg_Bicyclist', 'seg_Other Rider', 'seg_Wheeled Slow'])
df_with_composites['comp_motor_load']         = safe_sum(df_with_composites, ['seg_Car', 'seg_Truck', 'seg_Bus', 'seg_Motorcycle', 'seg_Other Vehicle'])
df_with_composites['comp_separation']         = safe_sum(df_with_composites, ['seg_Barrier', 'seg_Guard Rail', 'seg_Fence', 'seg_Wall', 'seg_Tunnel', 'seg_Bridge'])
df_with_composites['comp_natural_openness']   = safe_sum(df_with_composites, ['seg_Vegetation', 'seg_Terrain', 'seg_Mountain', 'seg_Water'])
df_with_composites['comp_commercial_density'] = safe_sum(df_with_composites, ['seg_Billboard', 'seg_Street Light', 'seg_Traffic Sign (Front)', 'seg_CCTV Camera'])
df_with_composites['comp_built_density']      = safe_sum(df_with_composites, ['seg_Building', 'seg_Wall', 'seg_Bridge'])

for suffix in ['cycling_infra', 'motor_load', 'separation', 'natural_openness', 'commercial_density', 'built_density']:
    df_with_composites[f'comp_has_{suffix}'] = (df_with_composites[f'comp_{suffix}'] > 0).astype(int)

df_with_composites['comp_nature_vs_built']  = df_with_composites['comp_natural_openness'] / (df_with_composites['comp_built_density'] + 0.001)
df_with_composites['comp_terrain_vs_motor'] = safe_sum(df_with_composites, ['seg_Terrain', 'seg_Vegetation']) / (df_with_composites['comp_motor_load'] + 0.001)

comp_cols = [c for c in df_with_composites.columns if c.startswith('comp_')]
print(f'Composite features added : {len(comp_cols)}')
print(f'Shape before cleaning    : {df_with_composites.shape[0]:,} rows x {df_with_composites.shape[1]:,} columns')

## 6. Feature cleaning (9 steps)

Identical to Stage 2. No changes to the cleaning pipeline.

In [ ]:
df            = df_with_composites.copy()
initial_shape = df.shape
_grid_re      = re.compile(r'_\d+_\d+$')
_all_raw      = [c for c in df.columns if c != 'image_name']
spatial_cols  = (
    [c for c in _all_raw if c.startswith('seg_') and c.endswith('_cx')] +
    [c for c in _all_raw if c.startswith('seg_') and c.endswith('_cy')] +
    [c for c in _all_raw if c.startswith('seg_') and c.endswith('_perimeter')]
)
area_cols    = [c for c in _all_raw if c.startswith('obj_') and '_area_' in c]
non_spatial  = [c for c in _all_raw if c not in spatial_cols]

# Step 1: remove unnamed columns
unnamed = [c for c in df.columns if str(c).startswith('Unnamed') or not str(c).strip()]
df      = df.drop(columns=unnamed) if unnamed else df
print(f'Step 1 — unnamed columns removed              : {len(unnamed)}')

# Step 2: extract point_id
df['point_id'] = df['image_name'].str.extract(r'^(\d+)_')
print(f'Step 2 — point_id extracted                   : {df["point_id"].nunique():,} unique locations')

# Step 3: selective NaN fill
non_spatial_present = [c for c in non_spatial if c in df.columns]
nan_non_spatial     = int(df[non_spatial_present].isna().sum().sum())
nan_spatial         = int(df[spatial_cols].isna().sum().sum()) if spatial_cols else 0
df[non_spatial_present] = df[non_spatial_present].fillna(0)
print(f'Step 3 — coverage/grid NaNs filled with 0     : {nan_non_spatial:,}')
print(f'         centroid/perimeter NaNs kept as NaN  : {nan_spatial:,}')

# Step 4: remove invalid all-zero rows
check_cols    = [c for c in non_spatial_present if not c.startswith('comp_')]
all_zero_mask = (df[check_cols] == 0).all(axis=1)
invalid_imgs  = df.loc[all_zero_mask, 'image_name'].tolist()
df            = df[~all_zero_mask].copy()
print(f'Step 4 — invalid all-zero rows removed        : {len(invalid_imgs)}')
for img in invalid_imgs:
    print(f'         {img}')

# Step 5: remove exact duplicate rows
dup_check    = [c for c in non_spatial_present if c in df.columns and not c.startswith('comp_')]
dup_mask     = df[dup_check].duplicated(keep='first')
removed_dups = df.loc[dup_mask, 'image_name'].tolist()
df           = df[~dup_mask].copy()
print(f'Step 5 — exact duplicate rows removed         : {len(removed_dups)}')

# Step 6: remove known sky-like duplicates
sky_dups   = ['2285_0.jpg', '2285_90.jpg', '2285_180.jpg', '2285_270.jpg',
              '2790_0.jpg', '2790_90.jpg', '2790_180.jpg', '2790_270.jpg']
before_sky = len(df)
df         = df[~df['image_name'].isin(sky_dups)].copy()
print(f'Step 6 — sky-like duplicates removed          : {before_sky - len(df)}')

# Step 7: normalise object area features
area_present    = [c for c in area_cols if c in df.columns]
area_max_before = df[area_present].max().max() if area_present else 0
if area_max_before > 1.0:
    df[area_present] = df[area_present] / IMAGE_PIXELS
    area_max_after   = df[area_present].max().max() if area_present else 0
    print(f'Step 7 — object area normalised               : {len(area_present)} columns')
    print(f'         max before: {area_max_before:,.1f}  ->  max after: {area_max_after:.6f}')
else:
    print(f'Step 7 — object area already normalised (max={area_max_before:.4f}) — skipped')

# Step 8: remove zero-variance columns
feat_cols     = [c for c in df.columns if c not in ('image_name', 'point_id')]
n_unique      = df[feat_cols].nunique(dropna=False)
zero_var_cols = n_unique[n_unique <= 1].index.tolist()
df            = df.drop(columns=zero_var_cols)
print(f'Step 8 — zero-variance columns removed        : {len(zero_var_cols)}')

# Step 9: remove near-zero columns
feat_cols    = [c for c in df.columns if c not in ('image_name', 'point_id')]
nonzero_frac = df[feat_cols].apply(lambda s: (s != 0).mean())
sparse_cols  = nonzero_frac[nonzero_frac < 0.001].index.tolist()
df           = df.drop(columns=sparse_cols)
print(f'Step 9 — near-zero columns removed            : {len(sparse_cols)} (< 0.1% non-zero)')

feat_cols_final = [c for c in df.columns if c not in ('image_name', 'point_id')]
features_df     = df.copy()
print()
print(f'Before cleaning : {initial_shape[0]:,} rows x {initial_shape[1]:,} columns')
print(f'After cleaning  : {len(features_df):,} rows x {len(feat_cols_final):,} features')
print(f'Locations       : {features_df["point_id"].nunique():,}')

## 7. Feature family definitions

Identical to Stage 2.

In [ ]:
def get_family(name):
    if name.startswith('comp_'):          return 'Composite'
    if name.startswith('seg_'):
        if _grid_re.search(name):         return 'Seg grid'
        elif name.endswith('_cx'):        return 'Seg centroid X'
        elif name.endswith('_cy'):        return 'Seg centroid Y'
        elif name.endswith('_perimeter'): return 'Seg perimeter'
        else:                             return 'Seg coverage'
    elif name.startswith('obj_'):
        if '_count_' in name:             return 'Obj count'
        elif '_area_' in name:            return 'Obj area'
        else:                             return 'Obj probability'
    else:
        return 'Image statistics'


print('Feature family definitions loaded.')

## 8. Shared functions

Identical to Stage 2. No changes.

In [ ]:
def bin_mu(series):
    return pd.qcut(series, q=NUM_BINS, labels=False, duplicates='drop')


def select_features_pearson(features_df_in, merged_df, mu_col, threshold):
    all_feats = [c for c in features_df_in.columns
                 if c not in ('image_name', 'point_id') and c in merged_df.columns]
    mu_values = merged_df[mu_col].values
    corr_vals = {}
    for feat in all_feats:
        x = merged_df[feat].replace([np.inf, -np.inf], np.nan).fillna(0).values
        corr_vals[feat] = 0.0 if x.std() == 0 else float(pearsonr(x, mu_values)[0])
    corr_df = (
        pd.DataFrame({'feature': list(corr_vals.keys()),
                      'pearson_r': list(corr_vals.values())})
          .assign(abs_r=lambda d: d['pearson_r'].abs())
          .sort_values('abs_r', ascending=False)
          .reset_index(drop=True)
    )
    selected = corr_df[corr_df['abs_r'] > threshold]['feature'].tolist()
    return selected, corr_df


def prep_data(features_df_in, votes_df, mu_col, sigma_col, threshold):
    merged = votes_df.merge(features_df_in, on='image_name', how='inner')
    merged['mu_bin'] = bin_mu(merged[mu_col])
    selected, corr_df = select_features_pearson(
        features_df_in, merged, mu_col, threshold
    )
    zv = [f for f in selected if merged[f].nunique() <= 1]
    if zv:
        selected = [f for f in selected if f not in zv]
    return merged, selected, corr_df


def get_sample_weights(sigma_values, y_values, class_weights):
    if SIGMA_WEIGHTING == 'none':
        sw = np.ones(len(sigma_values))
    elif SIGMA_WEIGHTING == '1/sigma':
        sw = 1.0 / np.maximum(sigma_values, 1e-6)
    elif SIGMA_WEIGHTING == '1/sigma^2':
        sw = 1.0 / np.maximum(sigma_values ** 2, 1e-12)
    else:
        raise ValueError(f'Unknown SIGMA_WEIGHTING: {SIGMA_WEIGHTING}')
    sw       = sw / sw.mean()
    cw       = np.array([class_weights[yi] for yi in y_values])
    combined = sw * cw
    return combined / combined.mean()


def preprocess_for_mlp(X_tr, X_val, X_te):
    imp    = SimpleImputer(strategy='constant', fill_value=0.5)
    scaler = StandardScaler()
    X_tr_i  = imp.fit_transform(X_tr)
    X_val_i = imp.transform(X_val)
    X_te_i  = imp.transform(X_te)
    X_tr_s  = scaler.fit_transform(X_tr_i)
    X_val_s = scaler.transform(X_val_i)
    X_te_s  = scaler.transform(X_te_i)
    return X_tr_s, X_val_s, X_te_s


print('Shared functions defined.')

## 9. Model definitions

Identical to Stage 2. The MLP uses the same fixed architecture with no additional tuning.
Tree model hyperparameters will be tuned in the next cell.

In [ ]:
def create_deeper_mlp(input_dim, num_classes=3):
    # fixed architecture identical to Stage 2:
    # 256 -> 128 -> 64 -> 32 -> output
    # first three layers: BatchNorm + ReLU + Dropout(0.3)
    # fourth layer: ReLU + Dropout(0.2)
    # output: softmax
    model = Sequential([
        Dense(256, input_dim=input_dim),
        BatchNormalization(),
        tf.keras.layers.Activation('relu'),
        Dropout(0.3),

        Dense(128),
        BatchNormalization(),
        tf.keras.layers.Activation('relu'),
        Dropout(0.3),

        Dense(64),
        BatchNormalization(),
        tf.keras.layers.Activation('relu'),
        Dropout(0.3),

        Dense(32),
        tf.keras.layers.Activation('relu'),
        Dropout(0.2),

        Dense(num_classes, activation='softmax'),
    ])
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


print('Model definitions loaded.')
print('  Deeper MLP : 256 -> 128 -> 64 -> 32 -> output (fixed, no tuning)')
print('  XGBoost    : hyperparameters tuned in next cell')
print('  LightGBM   : hyperparameters tuned in next cell')
print('  CatBoost   : hyperparameters tuned in next cell')

## 10. TrueSkill function

Identical to Stage 1. Draw probability is computed from the observed draw rate of each vote subset.

In [ ]:
def run_trueskill(vote_df, draw_prob, label):
    env     = trueskill.TrueSkill(draw_probability=draw_prob)
    ratings = {}

    for _, row in vote_df.iterrows():
        img1 = row['image_name1']
        img2 = row['image_name2']
        vote = row['vote']
        if img1 not in ratings: ratings[img1] = env.create_rating()
        if img2 not in ratings: ratings[img2] = env.create_rating()
        r1, r2 = ratings[img1], ratings[img2]
        if vote == 1:
            ratings[img1], ratings[img2] = env.rate_1vs1(r1, r2)
        elif vote == 2:
            ratings[img2], ratings[img1] = env.rate_1vs1(r2, r1)
        else:
            ratings[img1], ratings[img2] = env.rate_1vs1(r1, r2, drawn=True)

    records = [
        {'image_name': img, 'mu': r.mu, 'sigma': r.sigma,
         'ts_score': r.mu - 3 * r.sigma}
        for img, r in ratings.items()
    ]
    df_out = pd.DataFrame(records).sort_values('ts_score', ascending=False).reset_index(drop=True)
    df_out['rank'] = df_out['ts_score'].rank(ascending=False, method='min').astype(int)

    print(f'  {label}:')
    print(f'    Images ranked  : {len(df_out):,}')
    print(f'    Score range    : [{df_out["ts_score"].min():.4f}, {df_out["ts_score"].max():.4f}]')
    print(f'    Sigma range    : [{df_out["sigma"].min():.4f}, {df_out["sigma"].max():.4f}]')
    return df_out


print('TrueSkill function defined.')

## 11. Hyperparameter search for tree models

All three tree models are tuned once on the full 200k GPT ranking using GroupShuffleSplit
(20% location-level hold-out). The best parameters are then reused across all vote
thresholds for a fair comparison. This is identical to the tuning logic in Stage 2.

In [ ]:
# build full 200k ranking for hyperparameter search
print('Building full 200k TrueSkill ranking for hyperparameter search...')
draw_prob_200k   = (df_gpt_200k['vote'] == 0).mean()
gpt_rank_full_df = run_trueskill(df_gpt_200k, draw_prob_200k, 'GPT 200k (full)')
gpt_rank_full_df.columns = ['image_name', 'gpt_mu', 'gpt_sigma', 'gpt_ts_score', 'rank_gpt']

# feature selection on full 200k ranking
print('\nRunning feature selection on full 200k ranking...')
merged_full, feats_full, _ = prep_data(
    features_df, gpt_rank_full_df, 'gpt_mu', 'gpt_sigma', PEARSON_THRESHOLD_GPT
)
print(f'  Merged rows      : {len(merged_full):,}')
print(f'  Features selected: {len(feats_full)}')

X_full = merged_full[feats_full].reset_index(drop=True)
y_full = merged_full['mu_bin'].reset_index(drop=True)
s_full = merged_full['gpt_sigma'].reset_index(drop=True)
g_full = merged_full['point_id'].reset_index(drop=True)

# 20% location-level hold-out for tuning
gss       = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
tr_idx, _ = next(gss.split(X_full, y_full, groups=g_full))
X_s   = X_full.iloc[tr_idx]
y_s   = y_full.iloc[tr_idx]
sig_s = s_full.iloc[tr_idx]

cw_arr = compute_class_weight('balanced', classes=np.unique(y_s), y=y_s)
cw     = {i: w for i, w in enumerate(cw_arr)}
sw     = get_sample_weights(sig_s.values, y_s.values, cw)
scorer = make_scorer(balanced_accuracy_score)

print('\nTuning tree models on full 200k ranking...')

# XGBoost
xgb_search = RandomizedSearchCV(
    XGBClassifier(subsample=0.8, colsample_bytree=0.8,
                  random_state=SEED, eval_metric='mlogloss',
                  verbosity=0, use_label_encoder=False),
    {'n_estimators': [200, 300, 400], 'max_depth': [4, 6, 8],
     'learning_rate': [0.05, 0.10]},
    n_iter=N_ITER_SEARCH, scoring=scorer,
    cv=CV_SEARCH, random_state=SEED, n_jobs=-1, refit=False
)
xgb_search.fit(X_s.fillna(0), y_s, sample_weight=sw)
best_xgb_params = xgb_search.best_params_
print(f'  XGBoost  best: {best_xgb_params}  BA={xgb_search.best_score_:.4f}')

# LightGBM
lgb_search = RandomizedSearchCV(
    LGBMClassifier(subsample=0.8, colsample_bytree=0.8,
                   random_state=SEED, verbosity=-1),
    {'n_estimators': [200, 300, 400], 'max_depth': [4, 6, 8, -1],
     'learning_rate': [0.05, 0.10], 'num_leaves': [31, 63, 127]},
    n_iter=N_ITER_SEARCH, scoring=scorer,
    cv=CV_SEARCH, random_state=SEED, n_jobs=-1, refit=False
)
lgb_search.fit(X_s.fillna(0), y_s, sample_weight=sw)
best_lgb_params = lgb_search.best_params_
print(f'  LightGBM best: {best_lgb_params}  BA={lgb_search.best_score_:.4f}')

# CatBoost
cb_search = RandomizedSearchCV(
    CatBoostClassifier(random_state=SEED, verbose=0),
    {'iterations': [200, 300, 400], 'depth': [4, 6, 8],
     'learning_rate': [0.05, 0.10]},
    n_iter=N_ITER_SEARCH, scoring=scorer,
    cv=CV_SEARCH, random_state=SEED, n_jobs=-1, refit=False
)
cb_search.fit(X_s.fillna(0), y_s, sample_weight=sw)
best_cb_params = cb_search.best_params_
print(f'  CatBoost best: {best_cb_params}  BA={cb_search.best_score_:.4f}')

print('\nHyperparameter search complete. Parameters will be reused across all thresholds.')

## 12. CV runner functions

One function per model. All use the same 3x5 location-level GroupKFold protocol
as Stage 2. The MLP uses a 15% validation split within each training fold for
early stopping and learning rate reduction.

In [ ]:
def run_mlp_cv(X, y, sigma, groups):
    all_scores = []
    for part in range(N_PARTITIONS):
        sgkf       = StratifiedGroupKFold(n_splits=N_SPLITS)
        split_iter = list(sgkf.split(X, y, groups=groups))
        rng        = np.random.RandomState(SEED + part * 100)
        rng.shuffle(split_iter)
        for tr_idx, te_idx in split_iter:
            X_tr = X.iloc[tr_idx].values
            y_tr = y.iloc[tr_idx].values
            s_tr = sigma.iloc[tr_idx].values
            X_te = X.iloc[te_idx].values
            y_te = y.iloc[te_idx].values

            # validation split within training fold for early stopping
            n_val    = max(1, int(len(X_tr) * VAL_SIZE_WITHIN_TRAIN))
            X_val_f  = X_tr[-n_val:]
            y_val_f  = y_tr[-n_val:]
            X_tr_f   = X_tr[:-n_val]
            y_tr_f   = y_tr[:-n_val]
            s_tr_f   = s_tr[:-n_val]

            X_tr_s, X_val_s, X_te_s = preprocess_for_mlp(
                pd.DataFrame(X_tr_f),
                pd.DataFrame(X_val_f),
                pd.DataFrame(X_te)
            )

            cw_arr = compute_class_weight('balanced', classes=np.unique(y_tr_f), y=y_tr_f)
            cw     = {i: w for i, w in enumerate(cw_arr)}
            sw_tr  = get_sample_weights(s_tr_f, y_tr_f, cw)

            model = create_deeper_mlp(X_tr_s.shape[1], num_classes=NUM_BINS)
            callbacks = [
                EarlyStopping(monitor='val_loss', patience=PATIENCE,
                              restore_best_weights=True, verbose=0),
                ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                  patience=PATIENCE // 2, verbose=0)
            ]
            model.fit(
                X_tr_s, y_tr_f,
                validation_data=(X_val_s, y_val_f),
                epochs=EPOCHS, batch_size=BATCH_SIZE,
                sample_weight=sw_tr,
                callbacks=callbacks, verbose=0
            )
            y_pred = np.argmax(model.predict(X_te_s, verbose=0), axis=1)
            all_scores.append(balanced_accuracy_score(y_te, y_pred))
            tf.keras.backend.clear_session()
    return float(np.mean(all_scores)), float(np.std(all_scores))


def run_tree_cv(X, y, sigma, groups, clf_template):
    all_scores = []
    for part in range(N_PARTITIONS):
        sgkf       = StratifiedGroupKFold(n_splits=N_SPLITS)
        split_iter = list(sgkf.split(X, y, groups=groups))
        rng        = np.random.RandomState(SEED + part * 100)
        rng.shuffle(split_iter)
        for tr_idx, te_idx in split_iter:
            X_tr = X.iloc[tr_idx]; y_tr = y.iloc[tr_idx]; s_tr = sigma.iloc[tr_idx]
            X_te = X.iloc[te_idx]; y_te = y.iloc[te_idx]
            cw_arr = compute_class_weight('balanced', classes=np.unique(y_tr), y=y_tr)
            cw     = {i: w for i, w in enumerate(cw_arr)}
            sw_tr  = get_sample_weights(s_tr.values, y_tr.values, cw)
            clf    = sklearn.base.clone(clf_template)
            clf.fit(X_tr.fillna(0), y_tr, sample_weight=sw_tr)
            y_pred = clf.predict(X_te.fillna(0))
            all_scores.append(balanced_accuracy_score(y_te, y_pred))
    return float(np.mean(all_scores)), float(np.std(all_scores))


print('CV runner functions defined.')

## 13. Vote scaling loop — all four models

For each vote threshold:
1. Sample the first N votes from the combined 200k dataset (preserving order)
2. Compute TrueSkill ranking from the sampled votes
3. Merge with features and apply Pearson filter
4. Evaluate all four models with the same 3x5 GroupKFold protocol
5. Record mean BA and std for each model

The TrueSkill ranking is computed once per threshold and shared across all four models,
so there are 7 TrueSkill runs in total rather than 28.

In [ ]:
# build model templates using best tuned hyperparameters
xgb_template = XGBClassifier(
    **best_xgb_params,
    subsample=0.8, colsample_bytree=0.8,
    random_state=SEED, eval_metric='mlogloss',
    verbosity=0, use_label_encoder=False
)
lgb_template = LGBMClassifier(
    **best_lgb_params,
    subsample=0.8, colsample_bytree=0.8,
    random_state=SEED, verbosity=-1
)
cb_template = CatBoostClassifier(
    **best_cb_params,
    random_state=SEED, verbose=0
)

print('=' * 70)
print('VOTE SCALING ANALYSIS — ALL FOUR MODELS')
print('=' * 70)
print(f'Thresholds : {VOTE_THRESHOLDS}')
print(f'Protocol   : {N_PARTITIONS} x {N_SPLITS} = {N_PARTITIONS*N_SPLITS} evals per model per threshold')
print(f'Baseline   : {1/NUM_BINS*100:.1f}%')
print(f'Total CV runs: {len(VOTE_THRESHOLDS) * 4 * N_PARTITIONS * N_SPLITS}')
print()

all_scaling_results = []

for n_votes in VOTE_THRESHOLDS:
    print(f'=' * 60)
    print(f'THRESHOLD: {n_votes:,} votes')
    print(f'=' * 60)

    # 1. sample votes
    vote_subset  = df_gpt_200k.head(n_votes).copy()
    actual_votes = len(vote_subset)
    draw_prob    = (vote_subset['vote'] == 0).mean()
    print(f'  Votes used    : {actual_votes:,} | draw rate: {draw_prob*100:.2f}%')

    # 2. compute TrueSkill ranking (once per threshold, shared across all models)
    rank_df = run_trueskill(vote_subset, draw_prob, f'GPT {n_votes//1000}k')
    rank_df.columns = ['image_name', 'gpt_mu', 'gpt_sigma', 'gpt_ts_score', 'rank_gpt']

    # 3. merge with features and apply Pearson filter
    merged, feats, _ = prep_data(
        features_df, rank_df, 'gpt_mu', 'gpt_sigma', PEARSON_THRESHOLD_GPT
    )
    print(f'  Images merged : {len(merged):,} | Features: {len(feats)}')

    X      = merged[feats].reset_index(drop=True)
    y      = merged['mu_bin'].reset_index(drop=True)
    sigma  = merged['gpt_sigma'].reset_index(drop=True)
    groups = merged['point_id'].reset_index(drop=True)

    # 4. evaluate all four models
    results_this_threshold = {
        'n_votes'     : n_votes,
        'actual_votes': actual_votes,
        'n_images'    : len(merged),
        'n_features'  : len(feats),
    }

    print(f'  Running Deeper MLP...')
    mean_ba, std_ba = run_mlp_cv(X, y, sigma, groups)
    results_this_threshold['MLP_mean'] = mean_ba
    results_this_threshold['MLP_std']  = std_ba
    print(f'    Deeper MLP BA : {mean_ba:.4f} +/- {std_ba:.4f}')

    print(f'  Running XGBoost...')
    mean_ba, std_ba = run_tree_cv(X, y, sigma, groups, xgb_template)
    results_this_threshold['XGB_mean'] = mean_ba
    results_this_threshold['XGB_std']  = std_ba
    print(f'    XGBoost BA    : {mean_ba:.4f} +/- {std_ba:.4f}')

    print(f'  Running LightGBM...')
    mean_ba, std_ba = run_tree_cv(X, y, sigma, groups, lgb_template)
    results_this_threshold['LGB_mean'] = mean_ba
    results_this_threshold['LGB_std']  = std_ba
    print(f'    LightGBM BA   : {mean_ba:.4f} +/- {std_ba:.4f}')

    print(f'  Running CatBoost...')
    mean_ba, std_ba = run_tree_cv(X, y, sigma, groups, cb_template)
    results_this_threshold['CB_mean'] = mean_ba
    results_this_threshold['CB_std']  = std_ba
    print(f'    CatBoost BA   : {mean_ba:.4f} +/- {std_ba:.4f}')

    all_scaling_results.append(results_this_threshold)
    print()

scaling_df = pd.DataFrame(all_scaling_results)

print('Vote scaling analysis complete.')
print()
print(f'{"Votes":>10} {"Images":>8} {"Feats":>6} {"MLP":>10} {"XGB":>10} {"LGB":>10} {"CB":>10}')
print('-' * 70)
for _, row in scaling_df.iterrows():
    print(f'{int(row.n_votes):>10,} {int(row.n_images):>8,} {int(row.n_features):>6,} '
          f'{row.MLP_mean*100:>9.1f}% {row.XGB_mean*100:>9.1f}% '
          f'{row.LGB_mean*100:>9.1f}% {row.CB_mean*100:>9.1f}%')

## 14. Results chart — one curve per model

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

x_vals = scaling_df['n_votes'].values / 1000

model_specs = [
    ('Deeper MLP', 'MLP_mean',  'MLP_std',  'o',  MODEL_COLORS['Deeper MLP']),
    ('XGBoost',   'XGB_mean',  'XGB_std',  's',  MODEL_COLORS['XGBoost']),
    ('LightGBM',  'LGB_mean',  'LGB_std',  '^',  MODEL_COLORS['LightGBM']),
    ('CatBoost',  'CB_mean',   'CB_std',   'D',  MODEL_COLORS['CatBoost']),
]

y_all_max = 0
y_all_min = 1

for label, mean_col, std_col, marker, color in model_specs:
    y_means = scaling_df[mean_col].values
    y_stds  = scaling_df[std_col].values
    y_all_max = max(y_all_max, (y_means + y_stds).max())
    y_all_min = min(y_all_min, (y_means - y_stds).min())

    ax.plot(x_vals, y_means,
            color=color, linewidth=2.0,
            marker=marker, markersize=7,
            zorder=5, label=label)
    ax.fill_between(x_vals,
                    y_means - y_stds,
                    y_means + y_stds,
                    alpha=0.10, color=color, zorder=4)

# value labels at final threshold only to avoid clutter
for label, mean_col, std_col, marker, color in model_specs:
    y_means = scaling_df[mean_col].values
    y_stds  = scaling_df[std_col].values
    ax.text(x_vals[-1] + 1.5, y_means[-1],
            f'{y_means[-1]*100:.1f}%',
            ha='left', va='center',
            fontsize=8.5, fontweight='bold', color=color)

ax.set_xlabel('Number of GPT-5.2 synthetic votes (thousands)', fontsize=11)
ax.set_ylabel('Balanced accuracy (%)', fontsize=11)
ax.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda v, _: f'{v*100:.0f}%')
)
ax.set_xticks(x_vals)
ax.set_xticklabels([f'{int(x)}k' for x in x_vals], fontsize=9)
ax.set_xlim(x_vals[0] - 5, x_vals[-1] + 18)
ax.set_ylim(max(0.40, y_all_min - 0.02), y_all_max + 0.05)
ax.set_title(
    'Effect of synthetic vote volume on safety score prediction: all models\n'
    '3-class qcut | 1/\u03c3\u00b2 weighting | 3\u00d75 location-level GroupKFold | '
    'tree hyperparameters tuned on full 200k ranking',
    fontsize=10, fontweight='bold', pad=12, loc='left'
)
ax.legend(fontsize=9, frameon=True, framealpha=0.95,
          loc='lower right', edgecolor='#cccccc')
ax.spines[['top', 'right']].set_visible(False)
ax.spines['left'].set_color('#cccccc')
ax.spines['bottom'].set_color('#cccccc')
ax.grid(axis='y', alpha=0.2, linestyle='--', color='#aaaaaa')
ax.tick_params(labelsize=9)

plt.tight_layout()
plt.savefig('vote_scaling_all_models.pdf', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.savefig('vote_scaling_all_models.jpg', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.show()
print('Charts saved: vote_scaling_all_models.pdf and vote_scaling_all_models.jpg')